In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/src/small_bench/checkpoints/pre_cell_14.pickle

In [ ]:
%%cudf.pandas.profile
### cell 14 ###

### cell 14 optimized for cudf ###
# Compute first and last per Name on GPU for Mobile
mobile_grp   = Mobile_df.groupby('Name')
mobile_first = mobile_grp.first()
mobile_last  = mobile_grp.last()

Mobile_Stats = mobile_last[[
    'Avg. Avg D Kbps',
    'Avg. Avg U Kbps',
    'Avg Lat Ms'
]].copy()
Mobile_Stats['Change_Download'] = (
    mobile_last['Avg. Avg D Kbps'] - mobile_first['Avg. Avg D Kbps']
)
Mobile_Stats['Change_Upload'] = (
    mobile_last['Avg. Avg U Kbps'] - mobile_first['Avg. Avg U Kbps']
)
Mobile_Stats['Change_Latency'] = (
    mobile_last['Avg Lat Ms'] - mobile_first['Avg Lat Ms']
)
Mobile_Stats = Mobile_Stats[[
    'Change_Download',
    'Change_Upload',
    'Change_Latency'
]]

# Compute first and last per Name on GPU for Broadband
broad_grp   = Broadband_df.groupby('Name')
broad_first = broad_grp.first()
broad_last  = broad_grp.last()

Broadband_Stats = broad_last[[
    'Avg. Avg D Kbps',
    'Avg. Avg U Kbps',
    'Avg Lat Ms'
]].copy()
Broadband_Stats['Change_Download'] = (
    broad_last['Avg. Avg D Kbps'] - broad_first['Avg. Avg D Kbps']
)
Broadband_Stats['Change_Upload'] = (
    broad_last['Avg. Avg U Kbps'] - broad_first['Avg. Avg U Kbps']
)
Broadband_Stats['Change_Latency'] = (
    broad_last['Avg Lat Ms'] - broad_first['Avg Lat Ms']
)
Broadband_Stats = Broadband_Stats[[
    'Change_Download',
    'Change_Upload',
    'Change_Latency'
]]

# Combine Mobile and Broadband Download changes on GPU
Total_Stats = pd.concat([
    Mobile_Stats['Change_Download'].rename('Mobile'),
    Broadband_Stats['Change_Download'].rename('Broadband')
], axis=1)
